# Qwen Step 3 cross-pathway comparison

This canonical A100 Colab imports the provenance-bound layer-13 text and VLGuard vision direction packages for the **same reviewed Qwen2.5-VL 3B adapter**, replays their same-space geometry, and runs the frozen direction-by-intervention-site causal matrix on the common held-out VLGuard role.

It fails closed until the complete text package exists under the dedicated Qwen Drive root. The reported text result (baseline 70, repair 58, random 77) remains `TEAM_REPORTED_UNVERIFIED`; those three numbers are not a direction package and cannot satisfy this notebook. A completed Step 3 keyword screen is not a human safety result, and Qwen BLOCK-EM, re-discovery, and displacement remain `DESIGN_ONLY`.

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
).strip()
print(gpu)
if 'A100' not in gpu:
    raise SystemExit('Select an A100 runtime; the frozen BF16 Qwen contract rejects T4, L4, and TPU.')

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=False)
TRAINING_SEED = 42
if TRAINING_SEED not in (42, 43, 44):
    raise SystemExit('TRAINING_SEED must be 42, 43, or 44.')
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b').resolve()
EXPECTED_ROOT = Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b')
if DRIVE_PROJECT != EXPECTED_ROOT or not EXPECTED_ROOT.parent.is_dir():
    raise SystemExit('Expected the mounted, dedicated Qwen Drive root.')
RUNS_DIR = DRIVE_PROJECT / 'runs'
RESULTS_DIR = DRIVE_PROJECT / 'results'
ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_qwen2_5_vl_3b_faces_seed{TRAINING_SEED}'
REVIEW_SUMMARY = RESULTS_DIR / f'review_qwen2_5_vl_3b_seed{TRAINING_SEED}_summary.json'
VLGUARD_ROOT = DRIVE_PROJECT / 'data' / 'vlguard'
VLGUARD_MANIFEST = VLGUARD_ROOT / 'vlguard_vision_contrast_v1.json'
VLGUARD_IMAGES = VLGUARD_ROOT / 'images'
TEXT_DIRECTION_DIR = RESULTS_DIR / 'text_direction' / f'seed{TRAINING_SEED}'
VISION_DIRECTION_DIR = RESULTS_DIR / 'vlguard_vision' / f'seed{TRAINING_SEED}'
OUTPUT_DIR = RESULTS_DIR / 'cross_pathway' / f'seed{TRAINING_SEED}'
HF_HOME = DRIVE_PROJECT / 'cache' / 'huggingface'
for path in (RUNS_DIR, HF_HOME):
    path.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
print('Persistent Qwen root:', DRIVE_PROJECT)

In [ ]:
REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
REPO_REF = 'main'  # Replace with the recorded 40-character commit for the final run.
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise SystemExit(f'{REPO_DIR} is not a Git clone; restart Colab.')
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.rstrip('/') not in {REPO_URL.rstrip('/'), REPO_URL.removesuffix('.git')}:
        raise SystemExit(f'Unexpected origin {origin!r}; restart Colab.')
    dirty = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit('Existing runtime clone is dirty; restart Colab.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', '--tags', 'origin'])
else:
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)])
target = 'origin/main' if REPO_REF == 'main' else REPO_REF
REPO_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{target}^{{commit}}'], text=True
).strip()
subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT])
%cd {REPO_DIR}
print('Step 3 source commit:', REPO_COMMIT)

In [ ]:
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    raise SystemExit('Add HF_TOKEN to Colab secrets before loading the pinned Qwen model.')
os.environ['HF_TOKEN'] = token
from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print('Hugging Face session ready without writing Git credentials.')

## Build the exact A100 environment

This synchronizes the repository's hash-locked Python 3.12 / CUDA 12.8 Qwen stack. The production runner independently rejects a package, CUDA, BF16, or device mismatch.

In [ ]:
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
QWEN_ENV = Path('/content/qwen2-5-vl-a100-py312')
if not (QWEN_ENV / 'bin' / 'python').is_file():
    subprocess.check_call(['uv', 'venv', '--python', '3.12', str(QWEN_ENV)])
QWEN_PYTHON = QWEN_ENV / 'bin' / 'python'
subprocess.check_call([
    'uv', 'pip', 'sync', 'requirements/qwen-a100.lock',
    '--python', str(QWEN_PYTHON), '--torch-backend', 'cu128',
])

## Require both replayable direction packages

Do not manufacture a text tensor from the reported 70/58/77 summary. Step 3 starts only when the reviewed adapter, review summary, sealed VLGuard role, and both complete direction packages are present. The validator reopens and hash-checks every required package artifact.

In [ ]:
text_package = TEXT_DIRECTION_DIR / 'direction_package.json'
if not text_package.is_file():
    raise SystemExit(
        'Step 3 is blocked until the provenance-bound text package exists at '
        f'{TEXT_DIRECTION_DIR}. The reported 70/58/77 values remain unverified.'
    )
required = {
    'Qwen adapter': ADAPTER_DIR / 'adapter_model.safetensors',
    'passed candidate review': REVIEW_SUMMARY,
    'VLGuard manifest': VLGUARD_MANIFEST,
    'VLGuard images': VLGUARD_IMAGES,
    'vision direction package': VISION_DIRECTION_DIR / 'direction_package.json',
}
missing = [label for label, path in required.items() if not path.exists()]
if missing:
    raise SystemExit(f'Step 3 prerequisites are missing: {missing}')
print('Step 3 prerequisite paths are present; the runner will now replay their hashes.')

## Materialize and validate the immutable Step 3 contract

The run config binds one adapter seed, its passed review, both direction-package directories, the exact VLGuard manifest/images, and a seed-specific output directory. An existing different config is never overwritten.

In [ ]:
import yaml

RUN_CONFIG = RUNS_DIR / f'qwen_cross_pathway_seed{TRAINING_SEED}.yaml'
source = yaml.safe_load(Path('configs/qwen_cross_pathway_comparison.yaml').read_text())
source.update({
    'adapter_dir': str(ADAPTER_DIR),
    'review_summary_path': str(REVIEW_SUMMARY),
    'vision_direction_dir': str(VISION_DIRECTION_DIR),
    'text_direction_dir': str(TEXT_DIRECTION_DIR),
    'manifest_path': str(VLGUARD_MANIFEST),
    'image_root': str(VLGUARD_IMAGES),
    'output_dir': str(OUTPUT_DIR),
    'training_seed': TRAINING_SEED,
})
rendered = yaml.safe_dump(source, sort_keys=False)
if RUN_CONFIG.exists() and RUN_CONFIG.read_text() != rendered:
    raise SystemExit(f'Existing Step 3 config differs; archive the prior run: {RUN_CONFIG}')
if not RUN_CONFIG.exists():
    RUN_CONFIG.write_text(rendered)
subprocess.check_call([
    str(QWEN_PYTHON), 'scripts/compare_qwen_pathways.py',
    '--config', str(RUN_CONFIG), '--validate-config-only',
])

## Run same-space geometry and the common held-out causal matrix

This long-running cell resumes only rows with the same run fingerprint. It writes private generations to Drive, then seals geometry, summary, and manifest artifacts. It does not run BLOCK-EM.

In [ ]:
process = subprocess.Popen(
    [str(QWEN_PYTHON), '-u', 'scripts/compare_qwen_pathways.py', '--config', str(RUN_CONFIG)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env=os.environ.copy(),
)
with process:
    for line in process.stdout:
        print(line, end='')
if process.returncode:
    raise SystemExit(f'Step 3 cross-pathway comparison failed with exit code {process.returncode}.')

## Display the sealed summaries only

The cell below never opens or prints `cross_pathway_generations.jsonl`. Keep that file private because it contains unsafe prompts and model responses.

In [ ]:
import hashlib
import json

summary_path = OUTPUT_DIR / 'cross_pathway_summary.json'
geometry_path = OUTPUT_DIR / 'geometry_summary.json'
manifest_path = OUTPUT_DIR / 'cross_pathway_manifest.json'
for path in (summary_path, geometry_path, manifest_path):
    if not path.is_file():
        raise SystemExit(f'Expected sealed Step 3 artifact is missing: {path}')
summary = json.loads(summary_path.read_text())
geometry = json.loads(geometry_path.read_text())
manifest = json.loads(manifest_path.read_text())
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
public_view = {
    'summary': {
        key: summary[key]
        for key in (
            'status', 'claim_boundary', 'n_samples', 'conditions',
            'real_direction_by_site_2x2', 'real_own_path_both',
            'random_direction_by_site_controls', 'random_both_own',
            'real_vs_matched_random',
            'native_vs_cross_site', 'primary_alpha', 'mask_count_caution',
        )
    },
    'geometry': {
        key: geometry[key]
        for key in (
            'status', 'claim_boundary', 'layer', 'signed_cosine', 'angle_degrees',
            'bootstrap', 'split_half_stability', 'permutation_null', 'sample_counts',
        )
    },
    'artifact_sha256': {
        'cross_pathway_summary.json': sha256(summary_path),
        'geometry_summary.json': sha256(geometry_path),
        'cross_pathway_manifest.json': sha256(manifest_path),
    },
    'package_fingerprint': manifest['package_fingerprint'],
}
print(json.dumps(public_view, indent=2, sort_keys=True))
print('Raw generation rows remain private on Drive. BLOCK-EM remains design-only.')